## Step 1: Import Required Libraries

In [2]:
import pandas as pd
import numpy as np
import glob
import os

## Step 2: Load All Raw O3 Files (2020–2023)

In [12]:
o3_files = glob.glob("../../data/raw/O3_*.csv")

print("Files found:", o3_files)

o3_list = []

for file in o3_files:
    df = pd.read_csv(file, skiprows=7, encoding="latin1")
    o3_list.append(df)
    print("Loaded:", file, df.shape)

o3_raw = pd.concat(o3_list, ignore_index=True)

print("O3 combined shape:", o3_raw.shape)
o3_raw.head()

Files found: ['../../data/raw\\O3_2020.csv', '../../data/raw\\O3_2021.csv', '../../data/raw\\O3_2022.csv', '../../data/raw\\O3_2023.csv']
Loaded: ../../data/raw\O3_2020.csv (84912, 31)
Loaded: ../../data/raw\O3_2021.csv (84315, 31)
Loaded: ../../data/raw\O3_2022.csv (83950, 31)
Loaded: ../../data/raw\O3_2023.csv (85045, 31)
O3 combined shape: (338222, 31)


,Pollutant//Polluant,NAPS ID//Identifiant SNPA,City//Ville,Province/Territory//Province/Territoire,Latitude//Latitude,Longitude//Longitude,Date//Date,H01//H01,H02//H02,H03//H03,...,H15//H15,H16//H16,H17//H17,H18//H18,H19//H19,H20//H20,H21//H21,H22//H22,H23//H23,H24//H24
0,O3,10102,St. John's,NL,47.56038,-52.71147,2020-01-01,30,32,27,...,34,34,35,34,32,35,33,32,32,33
1,O3,10102,St. John's,NL,47.56038,-52.71147,2020-01-02,32,30,27,...,30,30,29,28,30,31,31,32,32,32
2,O3,10102,St. John's,NL,47.56038,-52.71147,2020-01-03,33,32,31,...,33,33,32,33,33,34,33,33,33,33
3,O3,10102,St. John's,NL,47.56038,-52.71147,2020-01-04,32,32,31,...,21,21,24,24,24,25,25,22,20,19
4,O3,10102,St. John's,NL,47.56038,-52.71147,2020-01-05,19,20,16,...,24,25,25,27,28,29,29,23,26,26


## Step 3: Clean Column Names
(Remove bilingual headers like H01//H01)


In [17]:
o3_raw.columns = [c.split("//")[0].strip() for c in o3_raw.columns]
print(o3_raw.columns)

Index(['Pollutant', 'NAPS ID', 'City', 'Province/Territory', 'Latitude',
       'Longitude', 'Date', 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07',
       'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17',
       'H18', 'H19', 'H20', 'H21', 'H22', 'H23', 'H24'],
      dtype='object')


## Step 4: Convert Date Column to Datetime

In [19]:
o3_raw["Date"] = pd.to_datetime(o3_raw["Date"], errors="coerce")

print("Invalid dates:", o3_raw["Date"].isna().sum())

Invalid dates: 0


## Step 5: Identify Hourly Columns (H01–H24)

In [22]:
hour_cols = [c for c in o3_raw.columns if c.startswith("H")]
print("Hour columns:", hour_cols)

Hour columns: ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17', 'H18', 'H19', 'H20', 'H21', 'H22', 'H23', 'H24']


## Step 6: Replace -999 with NaN

In [25]:
o3_raw[hour_cols] = o3_raw[hour_cols].replace(-999, np.nan)

## Step 7: Compute Daily O3 Average

In [28]:
o3_raw["O3_daily"] = o3_raw[hour_cols].mean(axis=1)

o3_raw[["City", "Date", "O3_daily"]].head()

,City,Date,O3_daily
0,St. John's,2020-01-01,32.625000
1,St. John's,2020-01-02,27.916667
2,St. John's,2020-01-03,32.166667
3,St. John's,2020-01-04,23.458333
4,St. John's,2020-01-05,19.208333


## Step 8: Create City-Day Level Dataset

In [31]:
o3_city = o3_raw[["City", "Date", "O3_daily"]].copy()

o3_city["Year"] = o3_city["Date"].dt.year
o3_city["Month"] = o3_city["Date"].dt.month

o3_city.head()

,City,Date,O3_daily,Year,Month
0,St. John's,2020-01-01,32.625000,2020,1
1,St. John's,2020-01-02,27.916667,2020,1
2,St. John's,2020-01-03,32.166667,2020,1
3,St. John's,2020-01-04,23.458333,2020,1
4,St. John's,2020-01-05,19.208333,2020,1


## Step 9: Remove Missing Daily Values

In [34]:
o3_city = o3_city.dropna(subset=["O3_daily"])

print("Remaining rows:", o3_city.shape)

Remaining rows: (327124, 5)


## Step 10: Remove Duplicates (City-Date Level)

In [37]:
o3_city = o3_city.drop_duplicates(subset=["City", "Date"])

print("Duplicates:", o3_city.duplicated(subset=["City", "Date"]).sum())

Duplicates: 0


## Step 11: Save Validated Daily Dataset

In [40]:
os.makedirs("../../data/validated", exist_ok=True)

o3_city.to_csv("../../data/validated/O3_cityday.csv", index=False)

print("Saved: O3_cityday.csv")

Saved: O3_cityday.csv
